In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_J.xpt"
csv_file = DATA_DIR / "HUQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 HUQ_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ041",
    "HUQ051",
    "HUQ061",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HUQ_J column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9254:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,254 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# HUQ_J contains no genuine continuous decimal variables.
#
# All fields are:
#
#     respondent identifier
#     questionnaire codes
#     whole-number count categories
#
# Therefore all 10 fields can ultimately become nullable Int64.
#
# IMPORTANT:
#
# This applies ONLY to HUQ_J.
#
# Genuine Dietary, Laboratory, Examination,
# sample-weight, or other true decimal variables in other
# NHANES files must remain untouched.
#
# No blanket rounding should ever be applied to those files.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 10:

    raise ValueError(
        "Expected exactly 10 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas decoding artifact:
#
#     5.397605346934028e-79
#
# This can represent numeric zero in some NHANES XPT files.
#
# NEVER globally replace it before validating the exact
# variable/count pattern against the CDC codebook.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 HUQ_J fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached HUQ_J.xpt:
#
# HUQ051 = 1,276 tiny values
#
# TOTAL  = 1,276
#
# CDC HUQ_J:
#
# HUQ051:
#
#     0 = None
#     Count = 1,276
#
# Therefore only these validated HUQ051 tiny values
# should be restored to numeric zero.
#

EXPECTED_TINY_COUNTS = {
    "HUQ051": 1276
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated HUQ_J attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 1276:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact HUQ051 1,276-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED HUQ051 ZEROS
# ============================================================

#
# DO NOT use:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# We restore only the validated HUQ051 observations.
#

huq051_tiny_mask = (
    df["HUQ051"]
    .eq(TINY_VALUE)
)


correction_count = int(
    huq051_tiny_mask.sum()
)


if correction_count != 1276:

    raise ValueError(
        "Unexpected HUQ051 tiny-value count."
    )


df.loc[
    huq051_tiny_mask,
    "HUQ051"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")

print(
    "HUQ051 restored to zero:",
    f"{correction_count:,}"
)


if int(
    df["HUQ051"]
    .eq(0)
    .sum()
) != 1276:

    raise ValueError(
        "HUQ051 zero restoration failed."
    )


print(
    "PASS: 1,276 validated HUQ051 zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# XPT-derived data should contain.
#
# The ONLY permitted numeric modification is:
#
#     HUQ051 tiny value -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ051"]
    .eq(TINY_VALUE),
    "HUQ051"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# IMPORTANT:
#
# This check occurs BEFORE converting anything to Int64.
#
# np.round() below is ONLY used for comparison.
# It does NOT modify the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in HUQ_J. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 12. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Every HUQ_J variable was checked above and proven to
# contain only whole-number values.
#
# There is NO .round() operation during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 HUQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. HELPER - EXACT CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. CDC CHECK - HUQ010
# ============================================================

check_frequency(
    "HUQ010",
    {
        1: 2303,
        2: 2501,
        3: 2847,
        4: 1335,
        5: 261,
        7: 5,
        9: 2
    },
    0
)


# ============================================================
# 15. CDC CHECK - HUQ020
# ============================================================

#
# Asked for participants age >= 1.
#

check_frequency(
    "HUQ020",
    {
        1: 1810,
        2: 654,
        3: 6427,
        9: 6
    },
    357
)


# ============================================================
# 16. CDC CHECK - HUQ030
# ============================================================

check_frequency(
    "HUQ030",
    {
        1: 7805,
        2: 1398,
        3: 49,
        9: 2
    },
    0
)


# ============================================================
# 17. CDC CHECK - HUQ041
# ============================================================

check_frequency(
    "HUQ041",
    {
        1: 1994,
        2: 5315,
        3: 291,
        4: 104,
        5: 137,
        6: 7,
        99: 6
    },
    1400
)


# ============================================================
# 18. HUQ041 ROUTING CHECK
# ============================================================

#
# HUQ041 is populated exactly when:
#
#     HUQ030 = 1
#
# OR
#
#     HUQ030 = 3
#

huq041_expected = (
    df["HUQ030"]
    .isin(
        [
            1,
            3
        ]
    )
    .to_numpy(
        dtype=bool
    )
)


huq041_present = (
    df["HUQ041"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


huq041_mismatch_count = int(
    np.sum(
        huq041_expected
        !=
        huq041_present
    )
)


print(
    "\n--- HUQ041 ROUTING ---"
)

print(
    "Routing mismatches:",
    huq041_mismatch_count
)


if huq041_mismatch_count != 0:

    raise ValueError(
        "HUQ041 routing does not match "
        "HUQ030 = 1 or 3."
    )


print(
    "PASS: HUQ041 routing validated."
)


# ============================================================
# 19. CDC CHECK - HUQ051
# ============================================================

check_frequency(
    "HUQ051",
    {
        0: 1276,
        1: 1945,
        2: 2932,
        3: 1421,
        4: 587,
        5: 266,
        6: 396,
        7: 128,
        8: 278,
        99: 25
    },
    0
)


# ============================================================
# 20. CDC CHECK - HUQ061
# ============================================================

check_frequency(
    "HUQ061",
    {
        1: 126,
        2: 113,
        3: 455,
        4: 366,
        5: 165,
        6: 45,
        99: 6
    },
    7978
)


# ============================================================
# 21. HUQ051 / HUQ061 ROUTING CHECK
# ============================================================

#
# HUQ061 is asked exactly when HUQ051 = 0.
#

huq051_none = (
    df["HUQ051"]
    .eq(0)
    .to_numpy(
        dtype=bool
    )
)


huq061_present = (
    df["HUQ061"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


huq061_mismatch_count = int(
    np.sum(
        huq051_none
        !=
        huq061_present
    )
)


print(
    "\n--- HUQ051 / HUQ061 ROUTING ---"
)

print(
    "Routing mismatches:",
    huq061_mismatch_count
)


if huq061_mismatch_count != 0:

    raise ValueError(
        "HUQ061 routing does not exactly "
        "match HUQ051=0."
    )


print(
    "PASS: All 1,276 HUQ051=None records "
    "route exactly to HUQ061."
)


# ============================================================
# 22. CDC CHECK - HUQ071
# ============================================================

check_frequency(
    "HUQ071",
    {
        1: 817,
        2: 8434,
        9: 3
    },
    0
)


# ============================================================
# 23. CDC CHECK - HUD080
# ============================================================

check_frequency(
    "HUD080",
    {
        1: 564,
        2: 150,
        3: 50,
        4: 21,
        5: 10,
        6: 17,
        99999: 5
    },
    8437
)


# ============================================================
# 24. HUQ071 / HUD080 ROUTING CHECK
# ============================================================

#
# HUD080 is populated exactly when HUQ071 = 1.
#

huq071_yes = (
    df["HUQ071"]
    .eq(1)
    .to_numpy(
        dtype=bool
    )
)


hud080_present = (
    df["HUD080"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hud080_mismatch_count = int(
    np.sum(
        huq071_yes
        !=
        hud080_present
    )
)


print(
    "\n--- HUQ071 / HUD080 ROUTING ---"
)

print(
    "Routing mismatches:",
    hud080_mismatch_count
)


if hud080_mismatch_count != 0:

    raise ValueError(
        "HUD080 routing does not exactly "
        "match HUQ071=1."
    )


print(
    "PASS: All 817 overnight-hospital records "
    "route exactly to HUD080."
)


# ============================================================
# 25. CDC CHECK - HUQ090
# ============================================================

#
# Asked for participants age >= 4.
#

check_frequency(
    "HUQ090",
    {
        1: 840,
        2: 7391,
        7: 2,
        9: 2
    },
    1019
)


# ============================================================
# 26. ALLOWED CDC CODE CHECK
# ============================================================

#
# Include CDC-documented codes even when their
# frequency in this cycle is zero.
#

ALLOWED_VALUES = {

    "HUQ010": {
        1, 2, 3, 4, 5, 7, 9
    },

    "HUQ020": {
        1, 2, 3, 7, 9
    },

    "HUQ030": {
        1, 2, 3, 7, 9
    },

    "HUQ041": {
        1, 2, 3, 4, 5, 6, 77, 99
    },

    "HUQ051": {
        0, 1, 2, 3, 4, 5, 6, 7, 8, 77, 99
    },

    "HUQ061": {
        1, 2, 3, 4, 5, 6, 77, 99
    },

    "HUQ071": {
        1, 2, 7, 9
    },

    "HUD080": {
        1, 2, 3, 4, 5, 6, 77777, 99999
    },

    "HUQ090": {
        1, 2, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    unexpected = (
        observed
        -
        allowed
    )

    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )

    print(
        f"PASS: {col} contains only "
        "CDC-documented values."
    )


# ============================================================
# 27. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original XPT contained zero ordinary numeric zeros.
#
# After repair, the ONLY zero values should be:
#
#     HUQ051 = 1,276
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated HUQ_J correction pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains exactly "
    "1,276 HUQ051 zeros."
)


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Only one numeric modification is allowed:
#
#     HUQ051:
#     1,276 tiny values -> 0
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match the exact "
    "expected corrected XPT values."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# IMPORTANT FIDELITY POLICY
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# Although HUQ_J itself contains no genuine decimals,
# keeping this export method is consistent with the
# broader NHANES workflow and avoids unnecessary
# decimal rewriting.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 33. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 35. RAW CSV .0 CHECK
# ============================================================

#
# HUQ_J contains NO genuine decimal fields.
#
# Therefore every HUQ_J field may safely be checked
# for unnecessary trailing ".0".
#
# This type of global .0 check must NOT be copied to a
# mixed file containing genuine decimal variables.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HUQ_J field."
)


# ============================================================
# 36. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "1,276 HUQ051 zeros."
)


# ============================================================
# 37. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 38. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 39. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "HUQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - "
    "Hospital Utilization & Access to Care"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "HUQ051 zero restorations:",
    correction_count
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "HUQ051 None:",
    int(
        df["HUQ051"]
        .eq(0)
        .sum()
    )
)

print(
    "HUQ071 overnight hospital = Yes:",
    int(
        df["HUQ071"]
        .eq(1)
        .sum()
    )
)

print(
    "HUQ090 mental health professional = Yes:",
    int(
        df["HUQ090"]
        .eq(1)
        .sum()
    )
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2017-2018 HUQ_J XPT -> CSV
Rows: 9,254
Columns: 10
PASS: Structure = 9,254 rows x 10 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: 10

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 1,276

Tiny values by variable:
HUQ051    1276

PASS: Exact HUQ051 1,276-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
HUQ051 restored to zero: 1,276
PASS: 1,276 validated HUQ051 zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HUQ010: 0 fractional observations
HUQ020: 0 fractional observations
HUQ030: 0 fractional observations
HUQ041: 0 fractional observations
HUQ051: 0 fractional observations
HUQ061: 0 fractional observations
HUQ071: 0 fractional observations
HUD080: 0 fractional observations
HUQ090: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimu